# Lab01S01 — Consulta GraphQL aos 100 repositórios mais populares do GitHub

| RQ | Pergunta | Métrica |
|:--:|----------|--------|
| 01 | Sistemas populares são maduros/antigos? | Idade do repositório |
| 02 | Recebem muita contribuição externa? | Total de PRs aceitas |
| 03 | Lançam releases com frequência? | Total de releases |
| 04 | São atualizados com frequência? | Tempo até última atualização |
| 05 | São escritos nas linguagens mais populares? | Linguagem primária |
| 06 | Possuem alto percentual de issues fechadas? | Razão issues fechadas / total |
| 07 | Sistemas em linguagens populares recebem mais contribuição, releases e updates? | PRs, releases e updates por grupo de linguagem |

## 1. Setup do Dataframe

In [ ]:
import pandas as pd
import glob
from datetime import datetime

In [11]:

# Busca o arquivo mais recente pelo timestamp no nome dentro da pasta data/
json_files = sorted(glob.glob("data/repos_lab01_s01*.json"), reverse=True)
csv_files = sorted(glob.glob("data/repos_lab01_s01*.csv"), reverse=True)

if json_files:
    latest = json_files[0]
    df = pd.read_json(latest, orient="records", convert_dates=["criado_em", "atualizado_em"])
    print(f"Carregado (JSON): {latest}")
elif csv_files:
    latest = csv_files[0]
    df = pd.read_csv(latest, parse_dates=["criado_em", "atualizado_em"])
    print(f"Carregado (CSV): {latest}")
else:
    raise FileNotFoundError("Nenhum arquivo repos_lab01_s01* encontrado em data/!")

print(f"Shape: {df.shape[0]} linhas x {df.shape[1]} colunas")

Carregado (JSON): data\repos_lab01_s0120260303_164918.json
Shape: 100 linhas x 12 colunas


## 2. Visão geral dos dados

In [12]:
df.describe()

,estrelas,idade_dias,dias_desde_update,prs_aceitas,releases,total_issues,issues_fechadas,razao_fechadas
count,100.000000,100.000000,100.0,100.000000,100.000000,100.000000,100.000000,100.000000
mean,163014.760000,3106.560000,0.0,7325.050000,144.890000,13516.400000,12024.120000,0.775094
std,83380.698073,1471.140946,0.0,13095.877922,249.238229,28620.295123,26258.088109,0.296334
min,96334.000000,99.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000
25%,107399.000000,2231.750000,0.0,389.000000,0.000000,476.000000,336.750000,0.712450
50%,131945.000000,3283.500000,0.0,1878.000000,19.500000,4027.500000,3392.500000,0.908900
75%,182192.750000,4214.250000,0.0,6946.250000,162.250000,14055.250000,12389.750000,0.971600
max,471702.000000,6031.000000,0.0,69505.000000,1000.000000,231789.000000,218510.000000,1.000000


## 3. Estatísticas por RQ (medianas)

In [13]:
medians = pd.Series({
    "RQ01 - Idade (dias)":               df["idade_dias"].median(),
    "RQ02 - PRs aceitas":                df["prs_aceitas"].median(),
    "RQ03 - Releases":                   df["releases"].median(),
    "RQ04 - Dias desde último update":   df["dias_desde_update"].median(),
    "RQ06 - Razão issues fechadas":      df["razao_fechadas"].median(),
}, name="Mediana")

medians.to_frame()

,Mediana
RQ01 - Idade (dias),3283.5000
RQ02 - PRs aceitas,1878.0000
RQ03 - Releases,19.5000
RQ04 - Dias desde último update,0.0000
RQ06 - Razão issues fechadas,0.9089


## 4. RQ05 — Sistemas populares são escritos nas linguagens mais populares?

Linguagens mais populares segundo o [GitHub Octoverse 2025](https://github.blog/news-insights/octoverse/octoverse-a-new-developer-joins-github-every-second-as-ai-leads-typescript-to-1/):
1. **TypeScript**
2. **Python**
3. **JavaScript**

In [14]:
LINGUAGENS_POPULARES = ["TypeScript", "Python", "JavaScript"]

lang_counts = df["linguagem"].fillna("N/A").value_counts()
print("Contagem por linguagem:")
print(lang_counts.to_frame("repositórios").to_string())

total_com_lang = df["linguagem"].notna().sum()
em_populares = df["linguagem"].isin(LINGUAGENS_POPULARES).sum()
pct = round(em_populares / total_com_lang * 100, 1)

print(f"\nRepositórios com linguagem definida: {total_com_lang}")
print(f"Repositórios em linguagens populares (Top 3 Octoverse): {em_populares} ({pct}%)")

Contagem por linguagem:
                  repositórios
linguagem                     
TypeScript                  18
Python                      18
N/A                         14
JavaScript                  11
C++                          6
Go                           5
HTML                         4
Rust                         4
Shell                        3
Java                         3
Markdown                     2
Dockerfile                   2
C#                           2
Jupyter Notebook             2
C                            2
Batchfile                    1
MDX                          1
Dart                         1
Vim Script                   1

Repositórios com linguagem definida: 86
Repositórios em linguagens populares (Top 3 Octoverse): 47 (54.7%)
